[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00


In [2]:
import torch
import math

In [14]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    d_k = K.shape[-1]
    QK = torch.bmm(Q, K.permute(0, 2, 1)) / math.sqrt(d_k)
    mask = torch.tril(torch.ones_like(QK)).bool()
    # print(mask)
    score = torch.where(mask, QK, -math.inf)
    return torch.bmm(torch.softmax(score, dim=-1), V)

In [15]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [16]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (22.1ms)
  ✅ [2/4] Future tokens don't affect past (9.0ms)
  ✅ [3/4] First position only sees itself (1.6ms)
  ✅ [4/4] Gradient flow (29.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (62.5ms total)
  Progress saved. Run status() to see your dashboard.

